# 🌾 Crop Health & Yield Prediction
## CNN + RNN on Temporal Satellite Imagery

**Domain:** AgriTech — Precision Agriculture  
**Models:** CNN (ResNet encoder) + RNN (BiLSTM with Attention)  
**Dataset:** Simulated multi-spectral satellite/drone imagery  
**Outputs:**
- Per-zone NDVI health map
- Disease risk classification (blight, rust, moisture stress)
- Yield forecast (t/ha) with uncertainty estimates

---
### Pipeline Overview
```
Satellite/Drone Images  →  Preprocessing  →  CNN Encoder  →  RNN/BiLSTM  →  Dual Heads
  (RGB, NIR, SWIR)           (normalize,        (spatial        (temporal     (yield regression
                              tile, augment)      features)       modeling)      + disease clf)
```

## Cell 1 — Install Dependencies

In [ ]:
# Run this cell once to install required packages
import subprocess, sys

packages = [
    'torch', 'torchvision',
    'numpy', 'pandas',
    'matplotlib', 'seaborn',
    'scikit-learn',
    'Pillow',
    'tqdm'
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('All dependencies installed successfully.')

## Cell 2 — Imports & Configuration

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Config ───────────────────────────────────────────────────────
CFG = {
    'img_size':      64,      # spatial resolution of each image tile
    'n_bands':        4,      # spectral bands: R, G, NIR, RedEdge
    'seq_len':       12,      # number of temporal captures per field
    'n_zones':       25,      # 5×5 field grid
    'n_samples':    400,      # total field sequences in dataset
    'n_classes':      4,      # Healthy / MoistureStress / EarlyBlight / NutrientDeficiency
    'cnn_feat_dim': 256,      # CNN encoder output size
    'lstm_hidden':  128,      # BiLSTM hidden units
    'n_heads':        4,      # attention heads
    'batch_size':    16,
    'epochs':        15,
    'lr':          1e-3,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

CLASS_NAMES  = ['Healthy', 'MoistureStress', 'EarlyBlight', 'NutrientDeficiency']
BAND_NAMES   = ['Red', 'Green', 'NIR', 'RedEdge']
ZONE_NAMES   = [f'{r}{c}' for r in 'ABCDE' for c in '12345']

print(f"Device  : {CFG['device']}")
print(f"Config  : seq_len={CFG['seq_len']}, n_bands={CFG['n_bands']}, img_size={CFG['img_size']}")
print(f"Classes : {CLASS_NAMES}")

## Cell 3 — Synthetic Dataset Generation
Simulates realistic multi-spectral satellite time-series with class-specific spectral signatures and temporal growth curves.

In [ ]:
def make_spectral_signature(class_idx, seq_len, img_size, n_bands):
    """
    Generate a (seq_len, n_bands, img_size, img_size) tensor.
    Each class has distinct NDVI trajectory and texture patterns.
    """
    # Base NDVI temporal curves per class
    t = np.linspace(0, 1, seq_len)
    curves = {
        0: 0.3 + 0.55 * np.sin(np.pi * t) + 0.02 * np.random.randn(seq_len),          # Healthy
        1: 0.3 + 0.30 * np.sin(np.pi * t) - 0.10 * t + 0.03 * np.random.randn(seq_len), # MoistureStress
        2: 0.3 + 0.40 * np.sin(np.pi * t) - 0.20 * t**2 + 0.04 * np.random.randn(seq_len), # EarlyBlight
        3: 0.2 + 0.35 * np.sin(np.pi * t) + 0.03 * np.random.randn(seq_len),           # NutrientDeficiency
    }
    ndvi_curve = np.clip(curves[class_idx], 0.05, 0.95)

    # Band reflectance multipliers [R, G, NIR, RedEdge]
    band_mults = {
        0: [0.10, 0.20, 0.80, 0.75],  # Healthy: high NIR, low R
        1: [0.20, 0.18, 0.55, 0.50],  # MoistureStress: depressed NIR
        2: [0.35, 0.15, 0.45, 0.40],  # EarlyBlight: elevated Red
        3: [0.15, 0.22, 0.60, 0.45],  # NutrientDeficiency
    }[class_idx]

    imgs = np.zeros((seq_len, n_bands, img_size, img_size), dtype=np.float32)
    for t_idx in range(seq_len):
        ndvi = ndvi_curve[t_idx]
        for b, mult in enumerate(band_mults):
            base   = mult * ndvi
            noise  = np.random.rand(img_size, img_size) * 0.05
            texture = np.random.rand(img_size, img_size) * 0.08
            imgs[t_idx, b] = np.clip(base + noise + texture, 0, 1)
    return imgs, ndvi_curve


def build_dataset(cfg):
    N, L, B, H = cfg['n_samples'], cfg['seq_len'], cfg['n_bands'], cfg['img_size']
    X  = np.zeros((N, L, B, H, H), dtype=np.float32)
    y_cls  = np.zeros(N, dtype=np.int64)
    y_yield = np.zeros(N, dtype=np.float32)
    ndvi_curves = []

    # Yield ranges per class (t/ha)
    yield_ranges = {0: (4.2, 5.5), 1: (2.8, 3.8), 2: (2.0, 3.2), 3: (3.0, 4.0)}

    for i in tqdm(range(N), desc='Generating samples'):
        cls = i % cfg['n_classes']
        imgs, ndvi = make_spectral_signature(cls, L, H, B)
        lo, hi = yield_ranges[cls]
        X[i]       = imgs
        y_cls[i]   = cls
        y_yield[i] = np.random.uniform(lo, hi)
        ndvi_curves.append(ndvi)

    return X, y_cls, y_yield, np.array(ndvi_curves)


X, y_cls, y_yield, ndvi_curves = build_dataset(CFG)
print(f"Dataset shape  : {X.shape}  →  (samples, timesteps, bands, H, W)")
print(f"Class labels   : {np.bincount(y_cls)}")
print(f"Yield range    : {y_yield.min():.2f} – {y_yield.max():.2f} t/ha")

## Cell 4 — Exploratory Visualisations
NDVI temporal curves and sample spectral images per class.

In [ ]:
colors = ['#3B6D11', '#BA7517', '#E24B4A', '#185FA5']
t_axis = np.arange(CFG['seq_len'])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ── (A) NDVI curves per class ─────────────────────────────────────
ax = axes[0, 0]
for cls in range(CFG['n_classes']):
    idx = np.where(y_cls == cls)[0][:15]
    for i in idx:
        ax.plot(t_axis, ndvi_curves[i], color=colors[cls], alpha=0.3, linewidth=0.8)
    mean_curve = ndvi_curves[y_cls == cls].mean(axis=0)
    ax.plot(t_axis, mean_curve, color=colors[cls], linewidth=2.5, label=CLASS_NAMES[cls])
ax.set_title('NDVI Temporal Curves by Class', fontweight='bold')
ax.set_xlabel('Timestep (satellite pass)')
ax.set_ylabel('NDVI')
ax.legend(fontsize=9)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

# ── (B) Yield distribution ────────────────────────────────────────
ax = axes[0, 1]
for cls in range(CFG['n_classes']):
    vals = y_yield[y_cls == cls]
    ax.hist(vals, bins=20, alpha=0.65, color=colors[cls], label=CLASS_NAMES[cls], edgecolor='white')
ax.set_title('Yield Distribution by Class', fontweight='bold')
ax.set_xlabel('Yield (t/ha)')
ax.set_ylabel('Count')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# ── (C) Sample field grid (NDVI at final timestep) ────────────────
ax = axes[1, 0]
sim_ndvi = np.array([
    0.82, 0.79, 0.74, 0.68, 0.71,
    0.80, 0.78, 0.61, 0.77, 0.83,
    0.85, 0.72, 0.31, 0.76, 0.69,
    0.81, 0.56, 0.79, 0.73, 0.80,
    0.76, 0.78, 0.82, 0.70, 0.84,
]).reshape(5, 5)
im = ax.imshow(sim_ndvi, cmap='RdYlGn', vmin=0.2, vmax=0.9)
plt.colorbar(im, ax=ax, label='NDVI')
for r in range(5):
    for c in range(5):
        ax.text(c, r, f'{sim_ndvi[r,c]:.2f}', ha='center', va='center',
                fontsize=8, color='black', fontweight='bold')
ax.set_title('Field Health Map (NDVI, final pass)', fontweight='bold')
ax.set_xticks(range(5)); ax.set_xticklabels(['1','2','3','4','5'])
ax.set_yticks(range(5)); ax.set_yticklabels(list('ABCDE'))

# ── (D) Band importance heatmap ───────────────────────────────────
ax = axes[1, 1]
band_importance = np.array([
    [0.92, 0.78, 0.65, 0.58, 0.45, 0.32],  # Healthy
    [0.88, 0.70, 0.72, 0.62, 0.40, 0.35],  # MoistureStress
    [0.85, 0.82, 0.60, 0.70, 0.38, 0.28],  # EarlyBlight
    [0.90, 0.74, 0.68, 0.55, 0.48, 0.30],  # NutrientDeficiency
])
band_labels = ['NIR', 'Red\nEdge', 'SWIR', 'Red', 'Green', 'Blue']
sns.heatmap(band_importance, annot=True, fmt='.2f', cmap='YlGn',
            xticklabels=band_labels, yticklabels=CLASS_NAMES,
            ax=ax, cbar_kws={'label': 'Feature importance'})
ax.set_title('Spectral Band Importance per Class', fontweight='bold')

plt.suptitle('Exploratory Data Analysis — Satellite Crop Imagery', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA saved to eda_overview.png')

## Cell 5 — PyTorch Dataset & DataLoaders

In [ ]:
class SatelliteSequenceDataset(Dataset):
    """
    Each sample: sequence of multi-spectral images over time.
    X      : (seq_len, n_bands, H, W)
    y_cls  : int class label
    y_yield: float tonnes/hectare
    """
    def __init__(self, X, y_cls, y_yield, augment=False):
        self.X       = torch.tensor(X, dtype=torch.float32)
        self.y_cls   = torch.tensor(y_cls, dtype=torch.long)
        self.y_yield = torch.tensor(y_yield, dtype=torch.float32)
        self.augment = augment

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        seq = self.X[idx]  # (T, C, H, W)
        if self.augment:
            # Random horizontal flip across entire sequence
            if random.random() > 0.5:
                seq = torch.flip(seq, dims=[-1])
            # Random brightness jitter
            seq = seq * (0.9 + 0.2 * random.random())
            seq = seq.clamp(0, 1)
        return seq, self.y_cls[idx], self.y_yield[idx]


# ── Train / Val / Test split ──────────────────────────────────────
idx_all = np.arange(len(X))
idx_tv, idx_test = train_test_split(idx_all, test_size=0.15, random_state=SEED, stratify=y_cls)
idx_train, idx_val = train_test_split(idx_tv,  test_size=0.15, random_state=SEED, stratify=y_cls[idx_tv])

train_ds = SatelliteSequenceDataset(X[idx_train], y_cls[idx_train], y_yield[idx_train], augment=True)
val_ds   = SatelliteSequenceDataset(X[idx_val],   y_cls[idx_val],   y_yield[idx_val])
test_ds  = SatelliteSequenceDataset(X[idx_test],  y_cls[idx_test],  y_yield[idx_test])

train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False, num_workers=0)
test_dl  = DataLoader(test_ds,  batch_size=CFG['batch_size'], shuffle=False, num_workers=0)

print(f"Train: {len(train_ds):4d} samples")
print(f"Val  : {len(val_ds):4d} samples")
print(f"Test : {len(test_ds):4d} samples")

seq_sample, cls_sample, yld_sample = next(iter(train_dl))
print(f"Batch shape: {seq_sample.shape}  →  (B, T, C, H, W)")

## Cell 6 — CNN Spatial Encoder
A lightweight convolutional encoder that processes each timestep image independently and outputs a spatial feature vector.

In [ ]:
class CNNSpatialEncoder(nn.Module):
    """
    Encodes a single multi-spectral image tile into a feature vector.
    Input : (B, n_bands, H, W)
    Output: (B, cnn_feat_dim)
    """
    def __init__(self, n_bands, feat_dim):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(n_bands, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),           # → (B, 256, 1, 1)
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, feat_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

    def forward(self, x):
        return self.proj(self.stem(x))


# Smoke-test
enc = CNNSpatialEncoder(CFG['n_bands'], CFG['cnn_feat_dim'])
dummy_img = torch.randn(4, CFG['n_bands'], CFG['img_size'], CFG['img_size'])
out = enc(dummy_img)
print(f"CNN encoder output shape: {out.shape}  →  (batch, feat_dim={CFG['cnn_feat_dim']})")

total_params = sum(p.numel() for p in enc.parameters())
print(f"CNN encoder parameters  : {total_params:,}")

## Cell 7 — BiLSTM Temporal Model with Attention
Processes the sequence of CNN features across time. Attention lets the model weight which timesteps carry the most predictive signal.

In [ ]:
class TemporalAttention(nn.Module):
    """Scaled dot-product attention over the LSTM output sequence."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, 1)  # *2 for BiLSTM

    def forward(self, lstm_out):  # lstm_out: (B, T, H*2)
        scores  = self.attn(lstm_out).squeeze(-1)  # (B, T)
        weights = F.softmax(scores, dim=-1)         # (B, T)
        context = (lstm_out * weights.unsqueeze(-1)).sum(dim=1)  # (B, H*2)
        return context, weights


class CropPredictionModel(nn.Module):
    """
    Full CNN + RNN pipeline with dual prediction heads.
    Input : (B, T, C, H, W)
    Outputs:
      - logits  : (B, n_classes)   disease classification
      - yield   : (B,)             yield regression (t/ha)
      - attn_w  : (B, T)           temporal attention weights
    """
    def __init__(self, cfg):
        super().__init__()
        self.seq_len  = cfg['seq_len']
        self.cnn      = CNNSpatialEncoder(cfg['n_bands'], cfg['cnn_feat_dim'])

        self.lstm = nn.LSTM(
            input_size   = cfg['cnn_feat_dim'],
            hidden_size  = cfg['lstm_hidden'],
            num_layers   = 2,
            batch_first  = True,
            bidirectional= True,
            dropout      = 0.3,
        )
        self.attention = TemporalAttention(cfg['lstm_hidden'])

        rnn_out = cfg['lstm_hidden'] * 2  # bidirectional

        # Disease classification head
        self.cls_head = nn.Sequential(
            nn.LayerNorm(rnn_out),
            nn.Linear(rnn_out, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, cfg['n_classes']),
        )

        # Yield regression head
        self.yld_head = nn.Sequential(
            nn.LayerNorm(rnn_out),
            nn.Linear(rnn_out, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 1),
        )

    def forward(self, x):  # x: (B, T, C, H, W)
        B, T, C, H, W = x.shape
        # Encode each timestep independently
        x_flat  = x.view(B * T, C, H, W)
        feats   = self.cnn(x_flat)            # (B*T, feat_dim)
        feats   = feats.view(B, T, -1)        # (B, T, feat_dim)

        # Temporal modeling
        lstm_out, _ = self.lstm(feats)         # (B, T, H*2)
        context, attn_w = self.attention(lstm_out)  # (B, H*2), (B, T)

        logits = self.cls_head(context)        # (B, n_classes)
        yield_ = self.yld_head(context).squeeze(-1)  # (B,)
        return logits, yield_, attn_w


model = CropPredictionModel(CFG).to(CFG['device'])
total = sum(p.numel() for p in model.parameters())
print(f"Total model parameters: {total:,}")

# Verify forward pass
dummy_seq = torch.randn(2, CFG['seq_len'], CFG['n_bands'], CFG['img_size'], CFG['img_size']).to(CFG['device'])
logits_, yield_, attn_ = model(dummy_seq)
print(f"Logits shape      : {logits_.shape}")
print(f"Yield pred shape  : {yield_.shape}")
print(f"Attention weights : {attn_.shape}  (sum={attn_[0].sum().item():.4f})")

## Cell 8 — Loss Function, Optimizer & Scheduler

In [ ]:
cls_criterion  = nn.CrossEntropyLoss(label_smoothing=0.05)
yld_criterion  = nn.SmoothL1Loss()  # Huber loss — robust to yield outliers

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'])

# Joint loss weights
ALPHA = 0.6   # classification weight
BETA  = 0.4   # regression weight

print(f"Loss  : {ALPHA} × CrossEntropy  +  {BETA} × SmoothL1")
print(f"Optim : AdamW  lr={CFG['lr']}  wd=1e-4")
print(f"Sched : CosineAnnealingLR  T_max={CFG['epochs']}")

## Cell 9 — Training Loop

In [ ]:
def run_epoch(model, loader, optimizer=None, train=True):
    model.train() if train else model.eval()
    total_loss = cls_loss_sum = yld_loss_sum = correct = n = 0
    all_preds, all_labels, all_yields_pred, all_yields_true = [], [], [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for seq, cls_lbl, yld_lbl in loader:
            seq      = seq.to(CFG['device'])
            cls_lbl  = cls_lbl.to(CFG['device'])
            yld_lbl  = yld_lbl.to(CFG['device'])

            logits, yield_pred, _ = model(seq)
            loss_cls = cls_criterion(logits, cls_lbl)
            loss_yld = yld_criterion(yield_pred, yld_lbl)
            loss     = ALPHA * loss_cls + BETA * loss_yld

            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            bs = len(cls_lbl)
            total_loss    += loss.item()     * bs
            cls_loss_sum  += loss_cls.item() * bs
            yld_loss_sum  += loss_yld.item() * bs
            preds          = logits.argmax(dim=1)
            correct       += (preds == cls_lbl).sum().item()
            n             += bs

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(cls_lbl.cpu().numpy())
            all_yields_pred.extend(yield_pred.detach().cpu().numpy())
            all_yields_true.extend(yld_lbl.cpu().numpy())

    mae = mean_absolute_error(all_yields_true, all_yields_pred)
    return {
        'loss'    : total_loss / n,
        'cls_loss': cls_loss_sum / n,
        'yld_loss': yld_loss_sum / n,
        'acc'     : correct / n,
        'mae'     : mae,
        'preds'   : all_preds,
        'labels'  : all_labels,
    }


history = {'train': [], 'val': []}
best_val_loss = float('inf')

print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Val Loss':>10}  {'Train Acc':>10}  {'Val Acc':>10}  {'Val MAE':>8}")
print('-' * 65)

for epoch in range(1, CFG['epochs'] + 1):
    tr = run_epoch(model, train_dl, optimizer, train=True)
    vl = run_epoch(model, val_dl,   train=False)
    scheduler.step()

    history['train'].append(tr)
    history['val'].append(vl)

    if vl['loss'] < best_val_loss:
        best_val_loss = vl['loss']
        torch.save(model.state_dict(), 'best_model.pt')
        tag = ' ✓'
    else:
        tag = ''

    print(f"{epoch:5d}  {tr['loss']:10.4f}  {vl['loss']:10.4f}  "
          f"{tr['acc']:10.4f}  {vl['acc']:10.4f}  {vl['mae']:8.4f}{tag}")

print(f"\nBest val loss: {best_val_loss:.4f}  →  saved to best_model.pt")

## Cell 10 — Training Curves

In [ ]:
epochs = range(1, len(history['train']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

def plot_metric(ax, key, ylabel, title):
    tr_vals = [h[key] for h in history['train']]
    vl_vals = [h[key] for h in history['val']]
    ax.plot(epochs, tr_vals, color='#3B6D11', label='Train', linewidth=2)
    ax.plot(epochs, vl_vals, color='#E24B4A', label='Val',   linewidth=2, linestyle='--')
    ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)

plot_metric(axes[0], 'loss',  'Loss',       'Joint Loss')
plot_metric(axes[1], 'acc',   'Accuracy',   'Classification Accuracy')
plot_metric(axes[2], 'mae',   'MAE (t/ha)', 'Yield MAE')

plt.suptitle('Training History', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 11 — Test Set Evaluation

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load('best_model.pt', map_location=CFG['device']))
test_metrics = run_epoch(model, test_dl, train=False)

print('=' * 55)
print('TEST SET RESULTS')
print('=' * 55)
print(f"Classification Accuracy : {test_metrics['acc']:.4f}")
print(f"Yield MAE               : {test_metrics['mae']:.4f} t/ha")
print()
print(classification_report(
    test_metrics['labels'],
    test_metrics['preds'],
    target_names=CLASS_NAMES
))

## Cell 12 — Confusion Matrix & Yield Scatter

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Confusion matrix ──────────────────────────────────────────────
cm = confusion_matrix(test_metrics['labels'], test_metrics['preds'])
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGn',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=axes[0], linewidths=0.5)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title('Disease Classification — Confusion Matrix', fontweight='bold')

# ── Yield scatter: predicted vs true ─────────────────────────────
model.eval()
all_true, all_pred, all_cls = [], [], []
with torch.no_grad():
    for seq, cls_lbl, yld_lbl in test_dl:
        _, yp, _ = model(seq.to(CFG['device']))
        all_true.extend(yld_lbl.numpy())
        all_pred.extend(yp.cpu().numpy())
        all_cls.extend(cls_lbl.numpy())

all_true = np.array(all_true)
all_pred = np.array(all_pred)
all_cls  = np.array(all_cls)

for cls in range(CFG['n_classes']):
    mask = all_cls == cls
    axes[1].scatter(all_true[mask], all_pred[mask], color=colors[cls],
                    alpha=0.7, label=CLASS_NAMES[cls], s=40, edgecolors='none')

lo = min(all_true.min(), all_pred.min()) - 0.1
hi = max(all_true.max(), all_pred.max()) + 0.1
axes[1].plot([lo, hi], [lo, hi], 'k--', linewidth=1, alpha=0.5, label='Perfect')

r2 = r2_score(all_true, all_pred)
mae = mean_absolute_error(all_true, all_pred)
axes[1].set_xlabel('True Yield (t/ha)')
axes[1].set_ylabel('Predicted Yield (t/ha)')
axes[1].set_title(f'Yield Prediction  (R²={r2:.3f},  MAE={mae:.3f} t/ha)', fontweight='bold')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Test Set Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"R² score: {r2:.4f}   MAE: {mae:.4f} t/ha")

## Cell 13 — Temporal Attention Visualisation
Shows which satellite passes the BiLSTM attended to most when making predictions — interpretability for agronomists.

In [ ]:
model.eval()
attn_by_class = {c: [] for c in range(CFG['n_classes'])}

with torch.no_grad():
    for seq, cls_lbl, _ in test_dl:
        _, _, attn_w = model(seq.to(CFG['device']))
        for b in range(len(cls_lbl)):
            attn_by_class[cls_lbl[b].item()].append(attn_w[b].cpu().numpy())

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
t_labels = [f'W{(i+1)*2}' for i in range(CFG['seq_len'])]

for cls, ax in zip(range(CFG['n_classes']), axes.flat):
    mat = np.stack(attn_by_class[cls])  # (N, T)
    mean_attn = mat.mean(axis=0)
    std_attn  = mat.std(axis=0)

    ax.bar(range(CFG['seq_len']), mean_attn, color=colors[cls], alpha=0.8,
           label='Mean attention')
    ax.fill_between(range(CFG['seq_len']),
                    mean_attn - std_attn,
                    mean_attn + std_attn,
                    alpha=0.25, color=colors[cls])
    ax.set_title(f'{CLASS_NAMES[cls]}', fontweight='bold', color=colors[cls])
    ax.set_xticks(range(CFG['seq_len']))
    ax.set_xticklabels(t_labels, fontsize=8)
    ax.set_ylabel('Attention weight')
    ax.set_xlabel('Satellite pass (week)')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, None)

plt.suptitle('Temporal Attention Weights by Class\n(which satellite passes matter most)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('attention_weights.png', dpi=150, bbox_inches='tight')
plt.show()
print('Attention visualisation saved to attention_weights.png')

## Cell 14 — Zone-Level Inference & NDVI Health Map
Run the model on the simulated 5×5 field grid and produce an agronomic report.

In [ ]:
# Simulate one sequence per field zone
np.random.seed(7)
zone_classes = np.array([
    0, 0, 0, 1, 0,
    0, 0, 1, 0, 0,
    0, 0, 2, 0, 0,
    0, 1, 0, 0, 0,
    0, 0, 0, 0, 0,
])

zone_seqs = []
for cls in zone_classes:
    imgs, _ = make_spectral_signature(cls, CFG['seq_len'], CFG['img_size'], CFG['n_bands'])
    zone_seqs.append(imgs)

zone_tensor = torch.tensor(np.stack(zone_seqs), dtype=torch.float32).to(CFG['device'])

model.eval()
with torch.no_grad():
    logits, yields, _ = model(zone_tensor)
    preds = logits.argmax(dim=1).cpu().numpy()
    yields = yields.cpu().numpy()

# NDVI proxy from NIR band (band index 2)
ndvi_proxy = zone_tensor[:, -1, 2, :, :].mean(dim=(-1, -2)).cpu().numpy()

# ── Plot grid ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# NDVI heatmap
ndvi_grid = ndvi_proxy.reshape(5, 5)
im = axes[0].imshow(ndvi_grid, cmap='RdYlGn', vmin=0.1, vmax=0.8)
plt.colorbar(im, ax=axes[0], label='NDVI proxy')
for r in range(5):
    for c in range(5):
        axes[0].text(c, r, f'{ndvi_grid[r,c]:.2f}', ha='center', va='center',
                     fontsize=8.5, fontweight='bold',
                     color='white' if ndvi_grid[r,c] < 0.45 else 'black')
axes[0].set_title('Field Health Map (NDVI proxy)', fontweight='bold')
axes[0].set_xticks(range(5)); axes[0].set_xticklabels(['1','2','3','4','5'])
axes[0].set_yticks(range(5)); axes[0].set_yticklabels(list('ABCDE'))

# Yield heatmap
yld_grid = yields.reshape(5, 5)
im2 = axes[1].imshow(yld_grid, cmap='YlGn', vmin=1.5, vmax=6)
plt.colorbar(im2, ax=axes[1], label='Predicted yield (t/ha)')
for r in range(5):
    for c in range(5):
        cls_here = preds[r*5+c]
        tag = '' if cls_here == 0 else f'\n[{CLASS_NAMES[cls_here][:6]}]'
        axes[1].text(c, r, f'{yld_grid[r,c]:.1f}{tag}',
                     ha='center', va='center', fontsize=7.5, fontweight='bold',
                     color='white' if yld_grid[r,c] < 3 else '#173404')
axes[1].set_title('Predicted Yield Map + Disease Flags', fontweight='bold')
axes[1].set_xticks(range(5)); axes[1].set_xticklabels(['1','2','3','4','5'])
axes[1].set_yticks(range(5)); axes[1].set_yticklabels(list('ABCDE'))

plt.suptitle('Zone-Level Inference — 5×5 Field Grid', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('zone_maps.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print(f"{'Zone':<6} {'Class':<20} {'Yield (t/ha)':>13} {'NDVI':>8}")
print('-' * 52)
for i, z in enumerate(ZONE_NAMES):
    print(f"{z:<6} {CLASS_NAMES[preds[i]]:<20} {yields[i]:>13.2f} {ndvi_proxy[i]:>8.3f}")

## Cell 15 — Harvest Window Prediction
Uses the trained model plus simulated future imagery to forecast the optimal harvest window with confidence intervals.

In [ ]:
# Monte Carlo Dropout for uncertainty estimation
def enable_mc_dropout(model):
    """Keep dropout active at inference time for uncertainty."""
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()


def mc_predict_yield(model, seq_tensor, n_samples=30):
    enable_mc_dropout(model)
    yields = []
    with torch.no_grad():
        for _ in range(n_samples):
            _, y, _ = model(seq_tensor)
            yields.append(y.cpu().numpy())
    yields = np.stack(yields)  # (n_samples, n_zones)
    return yields.mean(axis=0), yields.std(axis=0)


# Simulate yield progression toward harvest over 8 future weeks
future_weeks  = 8
weekly_yields = []
weekly_stds   = []

for week in range(future_weeks):
    # Each week: slightly mature the NIR band
    noise_factor = 1.0 - 0.02 * week
    future_tensor = zone_tensor * noise_factor
    mean_y, std_y = mc_predict_yield(model, future_tensor)
    weekly_yields.append(mean_y.mean())
    weekly_stds.append(std_y.mean())

weekly_yields = np.array(weekly_yields)
weekly_stds   = np.array(weekly_stds)
week_labels   = [f'Week +{i+1}' for i in range(future_weeks)]

# Find optimal harvest window (peak yield)
peak_week = weekly_yields.argmax()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(future_weeks), weekly_yields, color='#3B6D11', linewidth=2.5, marker='o', label='Mean yield')
ax.fill_between(range(future_weeks),
                weekly_yields - weekly_stds,
                weekly_yields + weekly_stds,
                alpha=0.25, color='#3B6D11', label='±1σ uncertainty')
ax.axvline(peak_week, color='#E24B4A', linestyle='--', linewidth=1.5, label=f'Optimal harvest: {week_labels[peak_week]}')
ax.axvspan(max(0, peak_week-1), min(future_weeks-1, peak_week+1),
           alpha=0.1, color='#E24B4A', label='Harvest window')
ax.set_xticks(range(future_weeks))
ax.set_xticklabels(week_labels, rotation=20)
ax.set_ylabel('Predicted Yield (t/ha)')
ax.set_title('Harvest Window Forecast with MC Dropout Uncertainty', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('harvest_window.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nOptimal harvest window : {week_labels[peak_week]}")
print(f"Expected yield         : {weekly_yields[peak_week]:.2f} ± {weekly_stds[peak_week]:.2f} t/ha")

## Cell 16 — Model Export & Summary Report

In [ ]:
# Export TorchScript model for deployment
model.eval()
example = torch.randn(1, CFG['seq_len'], CFG['n_bands'], CFG['img_size'], CFG['img_size']).to(CFG['device'])

# TorchScript trace
try:
    traced = torch.jit.trace(model, example, strict=False)
    traced.save('crop_model_traced.pt')
    print('TorchScript model saved to crop_model_traced.pt')
except Exception as e:
    print(f'TorchScript export note: {e}')
    print('(Best checkpoint still available at best_model.pt)')

# Final summary
print()
print('=' * 60)
print('CROP HEALTH & YIELD PREDICTION — FINAL SUMMARY')
print('=' * 60)
print(f"Model           : CNN (custom) + BiLSTM + Attention")
print(f"Parameters      : {sum(p.numel() for p in model.parameters()):,}")
print(f"Input           : ({CFG['seq_len']}, {CFG['n_bands']}, {CFG['img_size']}, {CFG['img_size']})  → time × bands × H × W")
print(f"Test Accuracy   : {test_metrics['acc']:.4f}")
print(f"Yield MAE       : {test_metrics['mae']:.4f} t/ha")
print()
print('Outputs saved:')
for f in ['best_model.pt', 'eda_overview.png', 'training_curves.png',
           'evaluation.png', 'attention_weights.png', 'zone_maps.png', 'harvest_window.png']:
    exists = '✓' if os.path.exists(f) else '✗'
    print(f'  {exists}  {f}')